# 05 — Feature Selection
**ChurnOps Phase 6** — Multi-Perspective Feature Evaluation & Redundancy Analysis  
*Combining Mutual Information, Variance Inflation Factors (VIF), Random Forest / XGBoost importances, Permutation Importance, and SHAP.*

---

### Phase 6 Objectives
1. **Information-Theoretic Utility:** Measure Mutual Information (MI) against `Churn` to evaluate non-linear dependency.
2. **Multicollinearity Diagnostic:** Calculate Variance Inflation Factors (VIF) and pairwise correlations to detect redundant regressors.
3. **Model-Based Feature Importance:** Compare Gini split importance (Random Forest) and gradient boost gain (XGBoost).
4. **Model-Agnostic Permutation Importance:** Scramble features on an unseen validation split to evaluate true generalization impact.
5. **SHAP Feature Attribution:** Analyze directional impact and magnitude of features using TreeExplainer.
6. **Feature Decision Matrix:** Produce a strict `Feature | Decision | Reason` table defining features to KEEP, DROP, or treat conditionally.

In [1]:
import sys, os
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from src.utils.config import get_path, PARAMS
from src.features.selection import (
    compute_mutual_information,
    compute_vif_scores,
    compute_model_and_permutation_importances
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

print("Project Root:", PROJECT_ROOT)

C:\Users\Abhishek\OneDrive\Pictures\Desktop\ChurnOps\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project Root: C:\Users\Abhishek\OneDrive\Pictures\Desktop\ChurnOps


In [2]:
data_path = PROJECT_ROOT / 'data' / 'interim' / 'telco_engineered.csv'
df = pd.read_csv(data_path)
print(f"Engineered Input Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
X = df.drop(columns=['Churn'])
y = df['Churn']

# One-hot encode for model algorithms
X_enc = pd.get_dummies(X, drop_first=True)
print(f"One-Hot Encoded Modeling Columns: {X_enc.shape[1]}")

Engineered Input Shape: 7,043 rows x 27 columns
One-Hot Encoded Modeling Columns: 39


## 1. Information-Theoretic Utility: Mutual Information

In [3]:
mi_series = compute_mutual_information(X_enc, y)
print("Top 10 Features by Mutual Information:")
print(mi_series.head(10))

plt.figure(figsize=(10, 5))
top_mi = mi_series.head(12)[::-1]
plt.barh(top_mi.index, top_mi.values, color='#3470a3', edgecolor='black', linewidth=0.5)
plt.title("Top 12 Features by Mutual Information with Churn", weight='bold')
plt.xlabel("Mutual Information")
for idx, (val, name) in enumerate(zip(top_mi.values, top_mi.index)):
    plt.annotate(f" {val:.4f}", (val, idx), va='center', fontsize=9)
plt.tight_layout()
plt.show()

Top 10 Features by Mutual Information:
is_month_to_month                      0.087732
tenure                                 0.074910
Contract_Two year                      0.068690
tenure_cohort_new                      0.053889
has_fiber_optic                        0.053007
MonthlyCharges                         0.050791
TotalCharges                           0.044219
StreamingMovies_No internet service    0.041090
PaymentMethod_Electronic check         0.039934
InternetService_Fiber optic            0.038627
dtype: float64


C:\Users\Abhishek\AppData\Local\Temp\ipykernel_23652\1168388505.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Multicollinearity & Variance Inflation Factor (VIF)

High VIF ($> 10$) signals that a feature can be linearly predicted from other predictors, which destabilizes linear model coefficients (Logistic Regression).

In [4]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'total_services']
vif_df = compute_vif_scores(df, num_cols)
corr_df = df[num_cols].corr()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

# VIF bar chart
bars = ax1.barh(vif_df['Feature'][::-1], vif_df['VIF'][::-1], color='#d95f02', edgecolor='black', linewidth=0.5)
ax1.axvline(10, color='red', linestyle='--', linewidth=1.5, label='Collinearity Threshold = 10')
ax1.set_title("Variance Inflation Factor (VIF)", weight='bold')
ax1.set_xlabel("VIF Value")
ax1.legend()
for p in bars:
    w = p.get_width()
    ax1.annotate(f" {w:.1f}", (w, p.get_y() + p.get_height()/2), va='center', weight='bold', fontsize=9)

# Correlation heatmap
sns.heatmap(corr_df, annot=True, fmt='.2f', cmap='vlag', vmin=-1, vmax=1, ax=ax2)
ax2.set_title("Pairwise Correlations", weight='bold')
plt.tight_layout()
plt.show()

vif_df

C:\Users\Abhishek\AppData\Local\Temp\ipykernel_23652\4278663709.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Feature,VIF
2,TotalCharges,10.27
0,tenure,5.85
3,total_services,4.42
1,MonthlyCharges,4.41


## 3. Model-Based & Permutation Importance

We train Random Forest and XGBoost on a train split and compute permutation importance (drop in ROC-AUC when a feature is shuffled) on an unseen validation split.

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X_enc, y, test_size=0.20, random_state=42, stratify=y)
rf_imp, xgb_imp, perm_df, shap_values, xgb_model = compute_model_and_permutation_importances(
    X_train, X_val, y_train, y_val
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

top_rf = rf_imp.head(10)[::-1]
ax1.barh(top_rf.index, top_rf.values, color='#2b7bba', edgecolor='black', linewidth=0.5)
ax1.set_title("Top 10 Random Forest Importances (Gini)", weight='bold')
ax1.set_xlabel("Importance")

top_perm = perm_df.head(10)[::-1]
ax2.barh(top_perm['Feature'], top_perm['Permutation_Mean'], color='#e6550d', edgecolor='black', linewidth=0.5)
ax2.set_title("Top 10 Permutation Importances (Validation ROC-AUC Drop)", weight='bold')
ax2.set_xlabel("Mean ROC-AUC Drop")

plt.tight_layout()
plt.show()

C:\Users\Abhishek\AppData\Local\Temp\ipykernel_23652\1125897173.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. SHAP (SHapley Additive exPlanations)

SHAP values attribute model output to each feature, showing both importance magnitude and directional impact on churn prediction.

In [6]:
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_val, show=False, max_display=12)
plt.title("SHAP Feature Summary (TreeExplainer on Validation Set)", weight='bold', pad=15)
plt.tight_layout()
plt.show()

C:\Users\Abhishek\AppData\Local\Temp\ipykernel_23652\4244673040.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Feature Decision Matrix

Based on our multi-perspective analysis, we establish our formal feature selection decisions:

| Feature | Decision | Justification |
|---------|----------|---------------|
| `is_month_to_month` | **KEEP** | Dominant churn predictor across all methods (Permutation Imp: 0.099, SHAP: 0.816). |
| `tenure` | **KEEP** | Primary continuous longevity signal (Permutation Imp: 0.067, SHAP: 0.430). |
| `has_fiber_optic` | **KEEP** | Flags highest-risk technological cohort (41.9% churn, SHAP: 0.273). |
| `MonthlyCharges` | **KEEP** | Core financial elasticity predictor (RF Imp: 0.109, SHAP: 0.239). |
| `PaymentMethod` | **KEEP** | High behavioral signal (Electronic check churn: 45.3% vs 15.2% for auto). |
| `has_security_or_techsupport` | **KEEP** | Substantial protective buffer (17.1% churn with support vs 33.4% without). |
| `total_services` | **KEEP** | Monotonic protective adoption indicator (RF Imp: 0.028). |
| `Contract` | **KEEP** | Granular multi-tier commitment (1-year and 2-year distinct retention tiers). |
| `InternetService` | **KEEP** | Captures DSL, Fiber, and No internet distinct baseline churn rates. |
| `PaperlessBilling` | **KEEP** | Statistically consistent moderate predictor (SHAP: 0.107). |
| Add-on services (`OnlineSecurity`, etc.) | **KEEP** | Retained for granular tree splits and prescriptive customer intervention. |
| Demographic flags (`SeniorCitizen`, `has_family`) | **KEEP** | Household stability predictors (14.2% family churn vs 30.6% solo). |
| `TotalCharges` | **CONDITIONAL** | VIF is 10.27 (high collinearity). Retained for tree models; monitored / L2-regularized in linear models. |
| `tenure_cohort` | **CONDITIONAL** | Binned steps beneficial for linear models; redundant for decision trees. |
| `gender` | **DROP** | Mutual Info = 0.000, Cramer's V = 0.008, Permutation Imp = 0.000. Male vs Female churn difference is 0.76% (pure noise). Dropping enforces algorithmic fairness and reduces variance. |
| `PhoneService` | **DROP** | Mutual Info = 0.000. 100% redundant with `MultipleLines` (which already contains `'No phone service'`). |

---

## 6. Phase 6 Summary & Hand-off to Phase 7
- **Evaluated:** 26 features across information theory, collinearity, tree importance, permutation importance, and SHAP.
- **Recommended Drops:** 2 features (`gender`, `PhoneService`).
- **Retained Features:** 24 features for pipeline modeling.
- **Leakage Integrity:** Evaluation performed on a dedicated validation split; any dynamic feature selection in downstream models will be embedded inside cross-validation folds.
- **Ready for:** Phase 7 — Train / Validation / Test Split (Stratified 80/20 split).